# Algorithmic Trading and Quantitative Strategies
## Part 9: Time Series Models and Statistical Arbitrage Foundations
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. [Stochastic Processes and Stationarity](#1.-Stochastic-Processes-and-Stationarity)
2. [Time Series Models](#2.-Time-Series-Models)
3. [Model Selection and Fitting](#3.-Model-Selection-and-Fitting)
4. [Cointegration](#4.-Cointegration)
5. [Introduction to Statistical Arbitrage](#5.-Introduction-to-Statistical-Arbitrage)
6. [Identifying Pairs](#6.-Identifying-Pairs)
7. [Exercises](#7.-Exercises)

## 1. Stochastic Processes and Stationarity

### 1.1 Stochastic Processes

A **stochastic process** $\{X_t\}$ is a collection of random variables indexed by time. Financial prices and returns are realizations of stochastic processes.

### 1.2 Stationarity

**Strict stationarity:** The joint distribution of $(X_{t_1}, X_{t_2}, \ldots, X_{t_k})$ is the same as $(X_{t_1+h}, X_{t_2+h}, \ldots, X_{t_k+h})$ for all $h$.

**Weak (covariance) stationarity:** A process is weakly stationary if:
1. $E[X_t] = \mu$ (constant mean)
2. $\text{Var}(X_t) = \sigma^2$ (constant variance)
3. $\text{Cov}(X_t, X_{t+h}) = \gamma(h)$ (autocovariance depends only on lag $h$)

**Why stationarity matters:**
- Statistical inference assumes stationarity
- Non-stationary processes (like prices) can produce **spurious regression** results
- Returns are (approximately) stationary; prices are not

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)

### 1.3 White Noise

The simplest stationary process: $\epsilon_t \sim WN(0, \sigma^2)$
- $E[\epsilon_t] = 0$
- $\text{Var}(\epsilon_t) = \sigma^2$
- $\text{Cov}(\epsilon_t, \epsilon_s) = 0$ for $t \neq s$

In [ ]:
# Generate and visualize white noise, random walk, and mean-reverting process
np.random.seed(42)
n = 500

# White noise
wn = np.random.normal(0, 1, n)

# Random walk: X_t = X_{t-1} + epsilon_t
rw = np.cumsum(np.random.normal(0, 1, n))

# Mean-reverting (Ornstein-Uhlenbeck): dX = theta*(mu - X)*dt + sigma*dW
theta, mu, sigma_ou = 0.5, 0, 0.5
ou = np.zeros(n)
for i in range(1, n):
    ou[i] = ou[i-1] + theta * (mu - ou[i-1]) + sigma_ou * np.random.normal()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(wn, linewidth=0.7)
axes[0].set_title('White Noise (Stationary)')
axes[0].axhline(y=0, color='r', linestyle='--')

axes[1].plot(rw, linewidth=0.7)
axes[1].set_title('Random Walk (Non-Stationary)')

axes[2].plot(ou, linewidth=0.7)
axes[2].set_title('Ornstein-Uhlenbeck (Mean-Reverting, Stationary)')
axes[2].axhline(y=mu, color='r', linestyle='--')

plt.tight_layout()
plt.show()

# ADF test for stationarity
for name, series in [('White Noise', wn), ('Random Walk', rw), ('OU Process', ou)]:
    result = adfuller(series)
    print(f"{name}: ADF stat = {result[0]:.4f}, p-value = {result[1]:.4f}, "
          f"{'Stationary' if result[1] < 0.05 else 'Non-Stationary'}")

## 2. Time Series Models

### 2.1 AR(p) — Autoregressive Model

$$X_t = c + \phi_1 X_{t-1} + \phi_2 X_{t-2} + \cdots + \phi_p X_{t-p} + \epsilon_t$$

- The current value depends on its own past values
- Stationary if all roots of the characteristic polynomial lie outside the unit circle
- For AR(1): stationary if $|\phi_1| < 1$
- **ACF:** Decays exponentially
- **PACF:** Cuts off after lag $p$

### 2.2 MA(q) — Moving Average Model

$$X_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + \cdots + \theta_q \epsilon_{t-q}$$

- Current value depends on past shocks
- Always stationary (finite linear combination of white noise)
- **ACF:** Cuts off after lag $q$
- **PACF:** Decays exponentially

### 2.3 ARMA(p,q) — Autoregressive Moving Average

$$X_t = c + \sum_{i=1}^{p} \phi_i X_{t-i} + \epsilon_t + \sum_{j=1}^{q} \theta_j \epsilon_{t-j}$$

### 2.4 Random Walk

$$X_t = X_{t-1} + \epsilon_t$$

This is AR(1) with $\phi_1 = 1$ — **non-stationary**. First differences $\Delta X_t = \epsilon_t$ are stationary (white noise).

**Random walk with drift:** $X_t = \mu + X_{t-1} + \epsilon_t$

In [ ]:
# Simulate AR(1), MA(1), ARMA(1,1) and examine ACF/PACF
np.random.seed(42)
n = 500

# AR(1): phi=0.7
ar1 = np.zeros(n)
for i in range(1, n):
    ar1[i] = 0.7 * ar1[i-1] + np.random.normal()

# MA(1): theta=0.7
ma1_noise = np.random.normal(size=n+1)
ma1 = np.array([ma1_noise[i] + 0.7 * ma1_noise[i-1] for i in range(1, n+1)])

fig, axes = plt.subplots(2, 3, figsize=(18, 8))

# AR(1)
axes[0, 0].plot(ar1[:100], 'b-')
axes[0, 0].set_title('AR(1), \u03c6=0.7')
plot_acf(ar1, lags=20, ax=axes[0, 1], title='ACF of AR(1)')
plot_pacf(ar1, lags=20, ax=axes[0, 2], title='PACF of AR(1)')

# MA(1)
axes[1, 0].plot(ma1[:100], 'r-')
axes[1, 0].set_title('MA(1), \u03b8=0.7')
plot_acf(ma1, lags=20, ax=axes[1, 1], title='ACF of MA(1)')
plot_pacf(ma1, lags=20, ax=axes[1, 2], title='PACF of MA(1)')

plt.tight_layout()
plt.show()

## 3. Model Selection and Fitting

### 3.1 ADF Test (Augmented Dickey-Fuller)

The ADF test checks for the presence of a unit root (non-stationarity):

$$\Delta X_t = \alpha + \beta t + \gamma X_{t-1} + \sum_{i=1}^{p} \delta_i \Delta X_{t-i} + \epsilon_t$$

- $H_0$: $\gamma = 0$ (unit root, non-stationary)
- $H_1$: $\gamma < 0$ (stationary)

### 3.2 Information Criteria

For ARMA model order selection:
- **AIC** = $-2\ln(L) + 2k$
- **BIC** = $-2\ln(L) + k\ln(n)$

Choose the model with the lowest AIC or BIC.

In [ ]:
# Download SPY returns for time series modeling
spy = yf.download('SPY', start='2015-01-01', end='2024-12-31')
spy.columns = spy.columns.droplevel('Ticker')
spy_returns = spy['Close'].pct_change().dropna()

print(f"SPY daily returns: {len(spy_returns)} observations")

# ADF test
adf_price = adfuller(spy['Close'].dropna())
adf_return = adfuller(spy_returns)

print(f"\nADF Test on SPY Prices:")
print(f"  Statistic: {adf_price[0]:.4f}, p-value: {adf_price[1]:.4f}")
print(f"  -> {'Stationary' if adf_price[1] < 0.05 else 'Non-Stationary'}")

print(f"\nADF Test on SPY Returns:")
print(f"  Statistic: {adf_return[0]:.4f}, p-value: {adf_return[1]:.4f}")
print(f"  -> {'Stationary' if adf_return[1] < 0.05 else 'Non-Stationary'}")

In [ ]:
# Fit ARMA models and compare AIC
from statsmodels.tsa.arima.model import ARIMA

results = []
for p in range(0, 4):
    for q in range(0, 4):
        try:
            model = ARIMA(spy_returns, order=(p, 0, q))
            fit = model.fit()
            results.append({'p': p, 'q': q, 'AIC': fit.aic, 'BIC': fit.bic})
        except:
            pass

results_df = pd.DataFrame(results).sort_values('AIC')
print("ARMA Model Comparison (top 10 by AIC):")
print(results_df.head(10).to_string(index=False))

# Fit best model
best = results_df.iloc[0]
print(f"\nBest model: ARMA({int(best['p'])},{int(best['q'])})")
print(f"AIC: {best['AIC']:.2f}, BIC: {best['BIC']:.2f}")

In [ ]:
# Auto ARIMA using pmdarima
try:
    from pmdarima import auto_arima
    
    auto_model = auto_arima(spy_returns, start_p=0, start_q=0, max_p=5, max_q=5,
                            seasonal=False, trace=True, suppress_warnings=True)
    print(f"\nAuto ARIMA selected: {auto_model.order}")
    print(auto_model.summary())
except ImportError:
    print("pmdarima not installed. Install with: pip install pmdarima")
    print("Alternatively, use manual AIC comparison above.")

## 4. Cointegration

### 4.1 Spurious Regression

Regressing one non-stationary series on another often produces a significant relationship even when none exists. This is called **spurious regression**.

### 4.2 Cointegration Definition

Two non-stationary series $X_t$ and $Y_t$ are **cointegrated** if there exists a linear combination:
$$Z_t = Y_t - \beta X_t$$
that is stationary. The coefficient $\beta$ is called the **cointegrating coefficient** or **hedge ratio**.

### 4.3 Engle-Granger Method

1. Estimate the cointegrating regression: $Y_t = \alpha + \beta X_t + \epsilon_t$
2. Test the residuals $\hat{\epsilon}_t$ for stationarity using ADF test
3. If residuals are stationary → series are cointegrated

In [ ]:
# Demonstrate cointegration with real data
# Download two related ETFs
tickers_coint = ['GLD', 'GDX']  # Gold ETF and Gold Miners ETF
prices_coint = yf.download(tickers_coint, start='2015-01-01', end='2024-12-31')['Close']
if isinstance(prices_coint.columns, pd.MultiIndex):
    prices_coint.columns = prices_coint.columns.droplevel(1)
prices_coint = prices_coint.dropna()

# Step 1: Check individual series for unit root
for col in prices_coint.columns:
    adf_result = adfuller(prices_coint[col])
    print(f"{col}: ADF stat = {adf_result[0]:.4f}, p-value = {adf_result[1]:.4f}")

# Step 2: Engle-Granger cointegration test
score, pvalue, _ = coint(prices_coint.iloc[:, 0], prices_coint.iloc[:, 1])
print(f"\nCointegration test:")
print(f"  Test statistic: {score:.4f}")
print(f"  p-value: {pvalue:.4f}")
print(f"  Cointegrated: {'Yes' if pvalue < 0.05 else 'No'}")

# Step 3: Estimate hedge ratio via OLS
Y = prices_coint.iloc[:, 1]
X = sm.add_constant(prices_coint.iloc[:, 0])
model = sm.OLS(Y, X).fit()
hedge_ratio = model.params.iloc[1]
intercept = model.params.iloc[0]
print(f"\nHedge ratio (\u03b2): {hedge_ratio:.4f}")
print(f"Intercept (\u03b1):   {intercept:.4f}")

# Spread (residuals)
spread = Y - hedge_ratio * prices_coint.iloc[:, 0] - intercept

# Test spread stationarity
adf_spread = adfuller(spread)
print(f"\nSpread ADF: stat = {adf_spread[0]:.4f}, p-value = {adf_spread[1]:.4f}")

In [ ]:
# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Prices
for col in prices_coint.columns:
    axes[0].plot(prices_coint[col], label=col)
axes[0].set_title(f'Prices: {tickers_coint[0]} vs {tickers_coint[1]}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Spread
axes[1].plot(spread, 'b-', linewidth=0.8)
axes[1].axhline(y=spread.mean(), color='r', linestyle='--', label='Mean')
axes[1].axhline(y=spread.mean() + 2*spread.std(), color='g', linestyle=':', label='\u00b12\u03c3')
axes[1].axhline(y=spread.mean() - 2*spread.std(), color='g', linestyle=':')
axes[1].set_title(f'Spread: {tickers_coint[1]} - {hedge_ratio:.2f}\u00d7{tickers_coint[0]}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Introduction to Statistical Arbitrage

### 5.1 History

Statistical arbitrage was pioneered by Nunzio Tartaglia's group at Morgan Stanley in the mid-1980s. They used statistical methods to identify pairs of stocks whose prices moved together, and traded the divergences.

### 5.2 Main Idea

Statistical arbitrage (stat arb) exploits **relative pricing** — the idea that similar securities should have similar prices. When prices diverge from their historical relationship, we:
1. **Buy** the relatively underpriced security
2. **Sell** the relatively overpriced security
3. **Wait** for convergence and close both positions

This is a **market-neutral** strategy — profit comes from the spread converging, not from market direction.

### 5.3 Steps
1. **Identify pairs:** Find securities with a stable long-run relationship
2. **Detect divergence:** Measure when the spread deviates from equilibrium
3. **Bet on convergence:** Open a long-short position
4. **Risk management:** Stop loss if divergence continues

## 6. Identifying Pairs

### 6.1 Same-Issuer Pairs
Different share classes of the same company (e.g., Voting vs Non-Voting)

### 6.2 Sector-Based Pairs
Companies in the same sector/industry (e.g., two banks, two airlines)

### 6.3 Distance Method (SSD)
Minimize the Sum of Squared Differences of normalized prices:
$$SSD = \sum_{t=1}^{T} (\hat{P}^A_t - \hat{P}^B_t)^2$$
where $\hat{P}$ denotes the normalized price (starting at 1).

### 6.4 Cointegration-Based Selection
Select pairs with statistically significant cointegration (low ADF p-value).

In [ ]:
# Systematic pair finding using cointegration
tickers_universe = ['XLF', 'XLK', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB']
prices_uni = yf.download(tickers_universe, start='2018-01-01', end='2024-12-31')['Close']
if isinstance(prices_uni.columns, pd.MultiIndex):
    prices_uni.columns = prices_uni.columns.droplevel(1)
prices_uni = prices_uni.dropna()

print(f"Testing cointegration for {len(tickers_universe)} ETFs "
      f"({len(tickers_universe)*(len(tickers_universe)-1)//2} pairs)")

# Test all pairs
pair_results = []
symbols = list(prices_uni.columns)
for i in range(len(symbols)):
    for j in range(i+1, len(symbols)):
        score, pvalue, _ = coint(prices_uni[symbols[i]], prices_uni[symbols[j]])
        pair_results.append({
            'Pair': f'{symbols[i]} / {symbols[j]}',
            'Asset1': symbols[i],
            'Asset2': symbols[j],
            'Coint_Stat': score,
            'p_value': pvalue,
        })

pairs_df = pd.DataFrame(pair_results).sort_values('p_value')
print("\nAll Pairs (sorted by p-value):")
print(pairs_df.to_string(index=False))

# Significant pairs
sig_pairs = pairs_df[pairs_df['p_value'] < 0.05]
print(f"\nSignificant pairs (p < 0.05): {len(sig_pairs)}")

In [ ]:
# Distance method
normalized = prices_uni / prices_uni.iloc[0]

# Compute SSD for all pairs
distance_results = []
for i in range(len(symbols)):
    for j in range(i+1, len(symbols)):
        ssd = ((normalized[symbols[i]] - normalized[symbols[j]])**2).sum()
        corr = prices_uni[symbols[i]].pct_change().corr(prices_uni[symbols[j]].pct_change())
        distance_results.append({
            'Pair': f'{symbols[i]} / {symbols[j]}',
            'SSD': ssd,
            'Correlation': corr,
        })

dist_df = pd.DataFrame(distance_results).sort_values('SSD')
print("Top 10 Closest Pairs by Distance (SSD):")
print(dist_df.head(10).to_string(index=False))

## 7. Exercises

1. **Stationarity Testing**: Download prices for AAPL and MSFT from 2015-2024. Test both price series and return series for stationarity using the ADF test.

2. **ARMA Modeling**: Fit ARMA models to SPY daily returns. Use AIC to select the best (p,q). Forecast the next 5 days of returns. How useful is the forecast?

3. **Cointegration Analysis**: Test for cointegration between:
   - KO and PEP (Coca-Cola and PepsiCo)
   - JPM and BAC (JPMorgan and Bank of America)
   - AAPL and MSFT (Apple and Microsoft)
   Which pairs are cointegrated?

4. **Pair Finding**: Download daily prices for all stocks in a sector (e.g., XLF components). Find the top 5 most cointegrated pairs.

5. **Spread Analysis**: For the best cointegrated pair from Exercise 3, compute the spread, plot it with Bollinger Bands (±2σ), and identify potential entry/exit signals.

---
### References
- Hamilton, J.D. (1994). *Time Series Analysis.* Princeton University Press.
- Tsay, R. (2010). *Analysis of Financial Time Series.* Wiley.
- Engle, R.F. & Granger, C.W.J. (1987). *Co-integration and Error Correction.* Econometrica.
- Vidyamurthy, G. (2004). *Pairs Trading.* Wiley.